## Zadanie domowe: BBHE i DSIHE

W klasycznym wyrównywaniu histogramu HE  po wykonaniu operacji jasność obrazu ulega zmianie.
Dało się to zaobserwować podczas przeprowadzonych eksperymentów.
Jeśli nie to należy uruchomić skrypt z sekcji A i zwrócić na to uwagę.
Średnia jasność dąży do środkowego poziomu szarości.
Jest to wada i dlatego klasyczne HE ma ograniczone zastosowanie.

Powstało sporo metod, które eliminują to niekorzystne zjawisko.
Najprostsze z nich polegają na dekompozycji obrazu wejściowego na dwa podobrazy (wg. pewnego kryterium).
Następnie operacja HE wykonywana jest dla tych podobrazów.

Dwie znane z literatury metody to:
- Bi-Histogram Equalization
- DSIHE - Dualistic Sub-Image Histogram Equalization

W metodzie BBHE za kryterium podziału przyjmuje się średnią jasność w obrazie.
W DSIHE obraz dzieli się na dwa podobrazy o takiej samej liczbie pikseli (jaśniejszych i ciemniejszych).

W ramach zadania należy zaimplementować wybraną metodę: BBHE lub DSIHE (ew. obie).

1. Wczytaj obraz *jet.bmp* i wylicz jego histogram.
2. W kolejnym kroku należy wyznaczyć próg podziału obrazu na dwa podobrazy (*lm*).
3. Dla BBHE wyznacz średnią jasność obrazu. Dla DSIHE można wykorzystać histogram skumulowany.
Należy znaleźć poziom jasności który znajduje się "w połowie" histogramu skumulowanego.
W tym celu warto stworzyć tablicę, zawierającą moduł histogramu skumulowanego pomniejszonego o połowę liczby pikseli.
Następnie znaleźć minimum.
4. Dalej należy podzielić histogram oryginalnego obrazu na dwa histogramy *H1* i *H2*.
Dla każdego z nich wyliczyć histogram skumulowany ($C_1$ i $C_2$) i wykonać normalizację.
Normalizacja polega na podzieleniu każdego histogramu przez jego największy element.
5. Na podstawie histogramów skumulowanych należy stworzyć przekształcenie LUT.
Należy tak przeskalować $C_1$ i $C_2$, aby uzyskać jednorodne przekształcenie.
Tablicę $C_1$ wystarczy pomnożyć przez próg podziału.
Tablicę $C_2$ należy przeskalować do przedziału: $<lm+1; 255>$, gdzie $lm$ jest progiem podziału.<br>
$C_{1n} = (lm)*C1;$<br>
$C_{2n} = lm+1 + (255-lm-1)*C2;$<br>
Następnie dwie części tablicy przekodowań należy połączyć.
6. Ostatecznie należy wykonać operację LUT i wyświetlić wynik wyrównywania histogramu.
Porównaj wynik operacji BBHE lub DSIHE z klasycznym HE.

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

import urllib.request

def download_if_not_exists(filename, url):
    if not os.path.exists(filename):
        print(f"Pobieranie {filename}...")
        urllib.request.urlretrieve(url, filename)

download_if_not_exists("jet.bmp","https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/jet.bmp")


img = cv2.imread("jet.bmp", cv2.IMREAD_GRAYSCALE)

print(img.shape)

hist = cv2.calcHist([img], [0], None, [256], [0, 256])

plt.plot(hist)
plt.title("Histogram obrazu")
plt.show()




In [ ]:
lm = int(np.mean(img))
print("Próg BBHE (średnia):", lm)


In [ ]:
H1 = hist[:lm+1]
H2 = hist[lm+1:]
C1 = np.cumsum(H1)
C2 = np.cumsum(H2)
C1 = C1 / C1[-1]
C2 = C2 / C2[-1]
C1n = (lm * C1).astype(np.uint8)
C2n = (lm + 1 + (255 - lm - 1) * C2).astype(np.uint8)
LUT = np.zeros(256, dtype=np.uint8)

LUT[:lm+1] = C1n
LUT[lm+1:] = C2n
result = cv2.LUT(img, LUT)
he = cv2.equalizeHist(img)
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("Oryginał")
plt.imshow(img, cmap='gray')

plt.subplot(1,3,2)
plt.title("Klasyczne HE")
plt.imshow(he, cmap='gray')

plt.subplot(1,3,3)
plt.title("BBHE ")
plt.imshow(result, cmap='gray')

plt.show()

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("Histogram oryginału")
plt.hist(img.flatten(), bins=256, range=[0,256])

plt.subplot(1,3,2)
plt.title("Histogram HE")
plt.hist(he.flatten(), bins=256, range=[0,256])

plt.subplot(1,3,3)
plt.title("Histogram BBHE")
plt.hist(result.flatten(), bins=256, range=[0,256])

plt.show()

print("Średnia jasność:")
print("Oryginał:", np.mean(img))
print("HE:", np.mean(he))
print("BBHE:", np.mean(result))